# Day1 - Ingestão de Exchange Rates via API

**Objetivo:** Consumir a API de taxas de câmbio (exchangeratesapi.io) e salvar o JSON bruto no Volume de Landing para posterior processamento na camada Bronze.

| Item | Valor |
|---|---|
| **Camada** | Landing (ingestão bruta via API REST) |
| **Entrada** | API `https://api.exchangeratesapi.io/v1/latest` |
| **Saída** | `/Volumes/{catalog}/landing/files/exchangerates/exchange_rates_{data}.json` |
| **Ordem no pipeline** | 1º passo — coleta de dados externos antes da carga Bronze |

> **⚠️ Atenção:** A chave da API é lida de um Databricks Secret Scope (`dbutils.secrets.get`). Nunca exponha a chave em texto plano no código ou em logs.

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Configuração da chamada à API

Importa bibliotecas necessárias e configura a URL da API. A chave de acesso é recuperada de forma segura via `dbutils.secrets.get` — **não** deve ser impressa ou logada.

In [0]:
import requests
import json
from datetime import datetime

base_url = 'https://api.exchangeratesapi.io/v1/'
endpoint = 'latest'
access_key = dbutils.secrets.get(scope="capgemini_academy", key="exchangerates_api_key")
data = datetime.now().strftime("%Y%m%d")

url = base_url + endpoint + '?access_key=' + access_key
print(f"Data da extração: {data}")
# NOTA: A URL completa NÃO é impressa para evitar expor a access_key em logs.

## Requisição HTTP

Executa a chamada GET à API e valida o status da resposta. Em caso de erro, exibe o código e o corpo da resposta para diagnóstico.

In [0]:
response = requests.get(url)

if response.status_code == 200:
    json_data = response.json()
    print(f"✓ Requisição bem-sucedida!")
    print(f"Dados recebidos: {list(json_data.keys())}\n")
else:
    print(f"✗ Erro na requisição: {response.status_code}")
    print(response.text)

## Persistência no Volume de Landing

Salva o JSON retornado pela API como arquivo no Volume do Unity Catalog. O nome do arquivo inclui a data da extração para permitir histórico.

In [0]:
volume_path = f"/Volumes/{catalog}/landing/files/exchangerates/"
file_name = f"exchange_rates_{data}.json"
full_path = f"{volume_path}/{file_name}"

try:
    with open(full_path, 'w') as f:
        json.dump(json_data, f, indent=2)
    print(f"✓ Arquivo salvo com sucesso!")
    print(f"Caminho: {full_path}")
except Exception as e:
    print(f"✗ Erro ao salvar: {e}")